Classical simulation methods:
- [Statevector (CPU, GPU)](https://qiskit.github.io/qiskit-aer/tutorials/1_aersimulator.html#Installing-GPU-Support)
- [Matrix product state (MPS)](https://qiskit.github.io/qiskit-aer/tutorials/7_matrix_product_state_method.html)
- [Operator backpropagation (OBP)](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-addons-obp-get-started)
- [Stabilizer circuit](https://quantum.cloud.ibm.com/docs/en/guides/simulate-stabilizer-circuits), [extended stabilizer](https://qiskit.github.io/qiskit-aer/tutorials/6_extended_stabilizer_tutorial.html)

In [155]:
import sys
from functools import partial
sys.path.append('../')
from math import log, comb, cos, sin
from numpy.linalg import matrix_power
from spd.OperatorSequence import *
from spd.SparsePauliDynamics import *
from spd.LightPauliDynamics import *
from pauli import *
from quantum_simulation_recipe.spin import Nearest_Neighbour_1d
from quantum_simulation_recipe.trotter import pf, expH
from qiskit.quantum_info import SparsePauliOp, random_statevector
import matplotlib.pyplot as plt

import numpy as np
from quantity import *
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector, SparsePauliOp, DensityMatrix, partial_trace, entropy
from qiskit_aer import AerSimulator, AerError
from qiskit_aer.library import save_statevector, save_density_matrix

from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_obp import backpropagate

fig_dir, data_dir = './figs', './data'

In [170]:
t = 5
r = 50
dt = t/r

n = 8
Jx, hx, hy = 1.0, 0.8, 0.9
# Jx, hx, hy = 1.0, 0.0, 0.9
qmfi = Nearest_Neighbour_1d(n, Jx=Jx, hx=hx, hy=hy, pbc=False)
xx_even = SparsePauliOp.from_sparse_list([*qmfi.xx_tuples[::2]], num_qubits=n).simplify()
xx_odd = SparsePauliOp.from_sparse_list([*qmfi.xx_tuples[1::2]], num_qubits=n).simplify()
x_terms = SparsePauliOp.from_sparse_list([*qmfi.x_tuples], num_qubits=n).simplify()
H_list = [qmfi.y_terms, x_terms, xx_even, xx_odd] # [xx_even, x_terms, qimf.y_terms,xx_odd]
print('Hamiltonian: \n', qmfi.ham)
init_state = Statevector.from_label('10'*int(n//2))
z1 = SparsePauliOp('I'*(n-1)+'Z', 1)
ob_op = z1
obs = ob_op.to_matrix()

pf1_xx_e = PauliEvolutionGate(xx_even, dt, synthesis=LieTrotter())
pf1_xx_o = PauliEvolutionGate(xx_odd, dt, synthesis=LieTrotter())
pf1_x = PauliEvolutionGate(x_terms, dt, synthesis=LieTrotter())
pf1_y = PauliEvolutionGate(qmfi.y_terms, dt, synthesis=LieTrotter())
gate_list = [pf1_y, pf1_x, pf1_xx_e, pf1_xx_o]

# We create an empty circuit
def create_circuit(n, gate_list, init_state=None, save_statevector=True, repeat=1, draw=-1):
    circuit = QuantumCircuit(n)
    if init_state is not None:
        circuit.set_statevector(init_state)
        # circuit.save_statevector(label='0')
    for d in range(repeat+1):
        for gate in gate_list:
            circuit.append(gate, range(n))
        if save_statevector:
            circuit.save_statevector(label=f'{d}')
    # if draw != -1 and n <= 6:
    #     circuit.decompose(reps=draw).draw('mpl')
    # circ_pf1_eo = transpile(circ_pf1_eo, optimization_level=0)
    return circuit

Hamiltonian: 
 SparsePauliOp(['IIIIIIXX', 'IIIIIXXI', 'IIIIXXII', 'IIIXXIII', 'IIXXIIII', 'IXXIIIII', 'XXIIIIII', 'IIIIIIIX', 'IIIIIIXI', 'IIIIIXII', 'IIIIXIII', 'IIIXIIII', 'IIXIIIII', 'IXIIIIII', 'XIIIIIII', 'IIIIIIIY', 'IIIIIIYI', 'IIIIIYII', 'IIIIYIII', 'IIIYIIII', 'IIYIIIII', 'IYIIIIII', 'YIIIIIII'],
              coeffs=[1. +0.j, 1. +0.j, 1. +0.j, 1. +0.j, 1. +0.j, 1. +0.j, 1. +0.j, 0.8+0.j,
 0.8+0.j, 0.8+0.j, 0.8+0.j, 0.8+0.j, 0.8+0.j, 0.8+0.j, 0.8+0.j, 0.9+0.j,
 0.9+0.j, 0.9+0.j, 0.9+0.j, 0.9+0.j, 0.9+0.j, 0.9+0.j, 0.9+0.j])


In [ ]:
## example
circ_pf1_eo = create_circuit(qmfi.n, gate_list, init_state=init_state, repeat=2)
# circ_pf1_eo.draw('mpl')
simulator1 = AerSimulator(method='statevector')
simulator2 = AerSimulator(method='statevector')
result1 = simulator1.run(circ_pf1_eo.decompose()).result()
result2 = simulator2.run(transpile(circ_pf1_eo)).result()
# result2 = simulator2.run(transpile(circ_pf1_eo, optimization_level=0)).result()
# print(result2.data(0)['statevector'])
print("check transpile:", np.vdot(result1.data(0)['1'], result2.data(0)['1'])**2)

check transpile: (0.9733200801484897+0.22945156695856753j)


## Stabilizer 

In [198]:
opts = {
    'extended_stabilizer_approximation_error': 0.03,
    'extended_stabilizer_mixing_time': 100
}
# sim_stab = AerSimulator(method='extended_stabilizer')
# res_stab = sim_stab.run(transpile(circ_pf1.decompose(), sim_stab), **opts).result().data()
sim_stab = AerSimulator(method='stabilizer')
res_stab = sim_stab.run(transpile(circ_pf1.decompose(), sim_stab)).result().data()
expvals_stab = [expect_value(obs, res_stab[str(d)].data) for d in range(r)]
fid_stab = [1-abs(np.vdot(qmfi_ideal_states[i+1], res_stab[str(i)].data))**2 for i in range(r)]
print(fid_stab)
t_list = [d*dt for d in range(r)]   
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(t_list, expvals_stab, label='Extended Stabilizer')
axes[0].plot(t_list, qmfi_ideal_expvals[1:], '--', label='Ideal')
axes[0].plot(t_list, qmfi_trott_expvals[1:], ':', label='Trotter')
axes[0].set_xlabel('Time Step')         

TranspilerError: 'Unable to translate the operations in the circuit: ["set_statevector", "save_statevector"] to the backend\'s (or manually specified) target basis: {"rz", "swap", "store", "sxdg", "for_loop", "pauli", "save_amplitudes_sq", "continue_loop", "save_probabilities_dict", "if_else", "set_stabilizer", "switch_case", "cy", "z", "save_state", "save_expval", "save_expval_var", "roerror", "cz", "s", "x", "h", "save_stabilizer", "cx", "reset", "save_probabilities", "ecr", "sdg", "qerror_loc", "quantum_channel", "measure", "save_clifford", "y", "sx", "break_loop", "delay", "while_loop", "id", "barrier", "snapshot"}. This likely means the target basis is not universal or there are additional equivalence rules needed in the EquivalenceLibrary being used. For more details on this error see: https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes. BasisTranslator#translation-errors'